# F1-scientific-python — Practice p24

**Type:** integrative · **Difficulty:** advanced · **Concepts:** seaborn-programming, random-seeding, aggregation-axis

**Set C — integration and challenge · Budget: 40 minutes.**

A teammate claims the revised process has higher scores, but the comparison below is not reproducible or auditable:

```python
rng = np.random.default_rng()
baseline = rng.normal(70, 8, 200)
revised = rng.normal(74, 8, 200)
sns.histplot(x=baseline)
sns.histplot(x=revised)
plt.axvline(revised.mean())
plt.show()
```

Complete all three parts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Part A — diagnosis

In 5–8 sentences, identify at least six concrete defects. Your diagnosis must cover reproducibility, comparable binning, axes ownership, labels/legend, overlap visibility, and the ambiguous mean marker.

The generator has no fixed seed, so the samples and every visual conclusion change between runs. Each histogram chooses bins independently, so bar heights are not comparisons over common intervals. The plotting calls rely on implicit current-axes state instead of creating and passing an owned `Axes`. Neither distribution has a label or legend, so the overlapping layers cannot be identified reliably. The default opaque overlap can hide one histogram behind the other, so a shared transparency setting is needed. The plot also lacks a title and axis labels, which makes the units and encoded quantity unclear. Finally, the single unlabeled mean line marks only the revised sample, so a reader cannot compare group means or even identify which mean it represents.

## Part B — reproducible repair

Implement exactly `def build_comparison():`. It must use `SEED = 20260804`, shared edges `np.linspace(40.0, 100.0, 13)`, one explicit `(7, 4)` axes, exactly two labeled `sns.histplot` calls with `alpha=0.45`, two dashed labeled mean markers, exact title and axis labels, a legend, and return the figure, axes, samples, and edges without `plt.show()`.

**Banned (zero points): unseeded randomness, separate bin-edge arrays, implicit current-axes plotting, and `plt.hist`.**

In [ ]:
def build_comparison():
    SEED = 20260804
    rng = np.random.default_rng(SEED)
    baseline = rng.normal(70, 8, 200)
    revised = rng.normal(74, 8, 200)
    shared_edges = np.linspace(40.0, 100.0, 13)

    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(x=baseline, bins=shared_edges, alpha=0.45, label="baseline", ax=ax)
    sns.histplot(x=revised, bins=shared_edges, alpha=0.45, label="revised", ax=ax)
    ax.axvline(baseline.mean(), linestyle="--", label="baseline mean")
    ax.axvline(revised.mean(), linestyle="--", label="revised mean")
    ax.set_title("Baseline and revised score distributions")
    ax.set_xlabel("score")
    ax.set_ylabel("count")
    ax.legend()
    return fig, ax, baseline, revised, shared_edges

Create `mean_baseline`, `mean_revised`, and `mean_shift` as Python floats, then run the immutable numerical and plot contract check.

In [ ]:
fig, ax, baseline, revised, shared_edges = build_comparison()
mean_baseline = float(baseline.mean())
mean_revised = float(revised.mean())
mean_shift = float(mean_revised - mean_baseline)

assert isinstance(mean_baseline, float) and isinstance(mean_revised, float)
assert isinstance(mean_shift, float)
assert ax.get_title() == "Baseline and revised score distributions"
assert ax.get_xlabel() == "score" and ax.get_ylabel() == "count"
assert np.allclose(mean_shift, mean_revised - mean_baseline, atol=1e-12, rtol=0)
expected_baseline_counts, _ = np.histogram(baseline, bins=shared_edges)
expected_revised_counts, _ = np.histogram(revised, bins=shared_edges)
patch_heights = np.array([patch.get_height() for patch in ax.patches])
n_bins = len(shared_edges) - 1
assert patch_heights.shape == (2 * n_bins,)
drawn_baseline_counts = patch_heights[:n_bins]
drawn_revised_counts = patch_heights[n_bins:]
assert np.array_equal(drawn_baseline_counts, expected_baseline_counts)
assert np.array_equal(drawn_revised_counts, expected_revised_counts)
plt.close(fig)

## Part C — justify the library boundary

In 4–6 sentences, explain why the distribution layers belong in `sns.histplot` while the exact mean markers belong in `ax.axvline`. State what the shared bins make comparable, what the summary numbers establish, and one limitation that the plot alone cannot resolve.

`sns.histplot` owns the distribution encoding because it converts observations into binned counts and draws the corresponding bars. `ax.axvline` is the appropriate lower-level axes method for an exact scalar reference position, so each sample mean is represented directly rather than rebinned. Shared edges ensure that corresponding bars count observations over identical score intervals. The three summary floats establish each sample mean and the signed revised-minus-baseline shift numerically. The plot and sample summaries alone cannot determine whether the process change caused the shift or whether it would persist in a new population.

### Answer check

The seeded samples, shared-bin counts, exact labels, two means, and Python-float summaries pass the immutable checks with `atol=1e-12` and `rtol=0`; the written diagnosis and justification cover all requested audit points.